# build_test_subset

Builds a smaller, more diverse test/example dataset by picking a handful of
BRUV stations out of a larger annotated dataset, then writing out a matching
COCO-style `instances_default.json` (filtered to just those stations) plus
the corresponding image files -- same folder layout as
`data/annotations/test_annotations`, so it's a drop-in replacement.

In [ ]:
# Chunk 0 - Auto-install missing packages

import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "yaml": "PyYAML",  # not used directly here, kept for parity with raw_to_yolo.ipynb
}

def install_missing_packages(packages: dict) -> None:
    """Check each required package can be imported; pip install any that are missing."""
    missing = []
    for import_name, pip_name in packages.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            missing.append(pip_name)

    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("All required packages are already installed.")

install_missing_packages(REQUIRED_PACKAGES)

In [ ]:
# Chunk 1 - Imports and shared path helper

import json
import random
import re
import shutil
from collections import defaultdict
from pathlib import Path


def find_repo_root(start: Path = None, marker: str = ".git") -> Path:
    """Walk upward from `start` until a directory containing `marker` is found."""
    start = start or Path.cwd()
    for directory in [start, *start.parents]:
        if (directory / marker).exists():
            return directory
    raise FileNotFoundError(
        f"Could not find repo root (looked for a parent directory containing '{marker}')."
    )


REPO_ROOT = find_repo_root()

In [ ]:
# Chunk 2 - Config
#
# SOURCE_ANNOTATIONS / SOURCE_IMAGES_DIR should point at your full, 80-station
# dataset (the CVAT/COCO export and its matching images/default folder).
# OUTPUT_DIR is where the new, smaller subset gets written -- same layout as
# data/annotations/test_annotations, so it can be pointed at directly from
# raw_to_yolo.ipynb's CONFIG afterward.

SOURCE_ANNOTATIONS = Path("/path/to/full_dataset/annotations/instances_default.json")  # EDIT ME
SOURCE_IMAGES_DIR = Path("/path/to/full_dataset/images/default")                       # EDIT ME
OUTPUT_DIR = REPO_ROOT / "data" / "annotations" / "test_subset"                        # EDIT ME if you want a different name

# Selection is by location (e.g. "Lovund" in
# NO_2025_3007_GarnRuse_Lovund_LO3_RA1_L9_frame_9), not individual BRUV
# station IDs -- every station at a selected location is included.
N_LOCATIONS = 10
SEED = 42

# Optional: skip random selection and hand-pick locations instead, e.g.
# MANUAL_LOCATIONS = ["Lovund", "Vandve", "Esjaholmen"]
MANUAL_LOCATIONS = None

In [ ]:
# Chunk 3 - Location / BRUV station helpers
# Same regex convention used in raw_to_yolo.ipynb, applied here to COCO
# `file_name` strings instead of Datumaro item IDs.

LOCATION_PATTERN = re.compile(r"GarnRuse_([A-Za-z]+)_")
DEPLOYMENT_PATTERN = re.compile(r"_([A-Z]{2,3}\d{1,2})_(?:RA|B)\d?_")


def extract_location(file_name: str) -> str:
    match = LOCATION_PATTERN.search(file_name)
    return match.group(1) if match else "Unknown"


def extract_bruv_id(file_name: str) -> str:
    match = DEPLOYMENT_PATTERN.search(file_name)
    if match:
        return match.group(1)
    parts = file_name.split("_")
    if len(parts) > 5:
        return parts[5]
    return extract_location(file_name)

In [ ]:
# Chunk 4 - Pick random locations

def select_random_locations(coco: dict, n_locations: int, seed: int = 42, manual_locations=None) -> list:
    """
    Randomly pick `n_locations` distinct locations (e.g. "Lovund", "Vandve")
    out of every location present in the dataset. Every BRUV station at a
    selected location is included in the resulting subset. Pass
    `manual_locations` to skip the random pick and hand-pick locations instead.
    """
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    location_stats = defaultdict(lambda: {"stations": set(), "n_images": 0, "n_crab": 0})
    for img in coco["images"]:
        loc = extract_location(img["file_name"])
        station = extract_bruv_id(img["file_name"])
        has_crab = len(anns_by_image[img["id"]]) > 0
        stats = location_stats[loc]
        stats["stations"].add(station)
        stats["n_images"] += 1
        stats["n_crab"] += int(has_crab)

    all_locations = sorted(location_stats.keys())
    total_stations = sum(len(s["stations"]) for s in location_stats.values())
    print(f"Full dataset: {len(all_locations)} locations, {total_stations} stations total.")

    if manual_locations:
        selected = [l for l in manual_locations if l in location_stats]
        missing = set(manual_locations) - set(selected)
        if missing:
            print(f"WARNING: requested locations not found in dataset: {sorted(missing)}")
    else:
        rng = random.Random(seed)
        selected = rng.sample(all_locations, min(n_locations, len(all_locations)))

    for loc in selected:
        stats = location_stats[loc]
        print(f"  {loc:<15} -> {len(stats['stations']):>2} stations, {stats['n_images']:>4} images, "
              f"{stats['n_crab']:>4} with crabs")

    return selected

In [ ]:
# Chunk 5 - Build the subset: filter the JSON and copy matching images

def build_test_subset(
    source_annotations: Path,
    source_images_dir: Path,
    output_dir: Path,
    n_locations: int,
    seed: int = 42,
    manual_locations=None,
) -> Path:
    print("=" * 100)
    print(f"Building a {n_locations}-location subset from {source_annotations}")

    with open(source_annotations, "r") as f:
        coco = json.load(f)

    selected_locations = select_random_locations(coco, n_locations, seed, manual_locations)
    print(f"\nSelected {len(selected_locations)} locations: {selected_locations}")

    keep_image_ids = {
        img["id"] for img in coco["images"]
        if extract_location(img["file_name"]) in selected_locations
    }

    subset_images = [img for img in coco["images"] if img["id"] in keep_image_ids]
    subset_annotations = [ann for ann in coco["annotations"] if ann["image_id"] in keep_image_ids]

    subset_coco = dict(coco)  # keep any extra top-level keys (e.g. "licenses") as-is
    subset_coco["images"] = subset_images
    subset_coco["annotations"] = subset_annotations

    out_ann_dir = output_dir / "annotations"
    out_img_dir = output_dir / "images" / "default"
    out_ann_dir.mkdir(parents=True, exist_ok=True)
    out_img_dir.mkdir(parents=True, exist_ok=True)

    out_json_path = out_ann_dir / "instances_default.json"
    with open(out_json_path, "w") as f:
        json.dump(subset_coco, f)

    missing_images = 0
    for img in subset_images:
        src = source_images_dir / img["file_name"]
        dst = out_img_dir / img["file_name"]
        if src.exists():
            shutil.copy2(src, dst)
        else:
            missing_images += 1

    n_crab_images = sum(1 for img in subset_images if len(
        [a for a in subset_annotations if a["image_id"] == img["id"]]
    ) > 0)

    subset_stations = {extract_bruv_id(img["file_name"]) for img in subset_images}
    print("=" * 100)
    print(f"Subset written to: {output_dir}")
    print(f"  Images: {len(subset_images)} ({n_crab_images} with at least one annotation)")
    print(f"  Annotations: {len(subset_annotations)}")
    print(f"  Locations: {len(selected_locations)} | Stations: {len(subset_stations)}")
    if missing_images:
        print(f"  WARNING: {missing_images} image files listed in the JSON were not found in {source_images_dir}")
    print("=" * 100)

    return out_json_path

In [ ]:
# Chunk 6 - Run it

build_test_subset(
    SOURCE_ANNOTATIONS,
    SOURCE_IMAGES_DIR,
    OUTPUT_DIR,
    n_locations=N_LOCATIONS,
    seed=SEED,
    manual_locations=MANUAL_LOCATIONS,
)